# Exploratory Data Analysis — Brugada-HUCA Dataset
**IDSC 2026 | Team MediScript**

This notebook explores the Brugada-HUCA dataset before modelling:
- Class distribution and imbalance
- Signal quality verification
- Sample ECG visualization per class
- Lead-wise amplitude statistics

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import wfdb

from config import DATA_DIR, METADATA_PATH
from src.preprocessing import merge_labels

plt.rcParams['figure.dpi'] = 120
print('Libraries loaded successfully.')

## 1. Load Metadata

In [ ]:
df = pd.read_csv(METADATA_PATH)
print(f'Total records: {len(df)}')
print(f'\nRaw label distribution:')
print(df['brugada'].value_counts())
print(f'\nColumns: {list(df.columns)}')
df.head()

## 2. Class Distribution

The dataset has a 3.37:1 imbalance (Normal:Brugada). Seven borderline cases 
(label 2) were merged into the positive class — 4/7 showed spontaneous Brugada 
ECG patterns and 1/7 experienced sudden cardiac death, making them clinically 
high-risk.

In [ ]:
df['label_merged'] = merge_labels(df['brugada'].tolist())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

raw_counts = df['brugada'].value_counts().sort_index()
axes[0].bar(
    ['Normal (0)', 'Brugada (1)', 'Borderline (2)'],
    [raw_counts.get(0,0), raw_counts.get(1,0), raw_counts.get(2,0)],
    color=['steelblue', 'crimson', 'orange'], edgecolor='white'
)
axes[0].set_title('Raw Label Distribution', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate([raw_counts.get(0,0), raw_counts.get(1,0), raw_counts.get(2,0)]):
    axes[0].text(i, v + 2, str(v), ha='center', fontweight='bold')

merged_counts = df['label_merged'].value_counts().sort_index()
axes[1].bar(
    ['Normal (0)', 'Brugada (1)'],
    [merged_counts.get(0,0), merged_counts.get(1,0)],
    color=['steelblue', 'crimson'], edgecolor='white'
)
axes[1].set_title('After Merging Label 2 → 1', fontweight='bold')
axes[1].set_ylabel('Count')
for i, v in enumerate([merged_counts.get(0,0), merged_counts.get(1,0)]):
    axes[1].text(i, v + 2, str(v), ha='center', fontweight='bold')
ratio = merged_counts.get(0,0) / merged_counts.get(1,0)
axes[1].text(0.5, 0.85, f'Imbalance ratio: {ratio:.2f}:1',
             transform=axes[1].transAxes, ha='center', color='gray')

plt.suptitle('Brugada-HUCA Class Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Signal Quality Verification

Check that all 363 records load without errors, have consistent shape 
(1200 samples × 12 leads), and contain no NaN values.

In [ ]:
print('Checking all records — this takes about 1 minute...')
shapes, nan_count, failed = [], 0, []

for pid in df['patient_id'].astype(str):
    try:
        rec = wfdb.rdrecord(f'{DATA_DIR}/{pid}/{pid}')
        shapes.append(rec.p_signal.shape)
        if np.isnan(rec.p_signal).any():
            nan_count += 1
    except Exception as e:
        failed.append((pid, str(e)))

print(f'Successfully loaded : {len(df) - len(failed)}/{len(df)}')
print(f'Failed              : {len(failed)}')
print(f'Records with NaN    : {nan_count}')
print(f'Unique signal shapes: {set(shapes)}  ← should be only (1200, 12)')

## 4. Sample ECG Visualization — Normal vs. Brugada

Look at all 12 leads side by side. Pay attention to leads V1, V2, V3 — 
the diagnostic hot zone for Brugada syndrome. The coved ST-segment elevation 
in V1–V3 is the hallmark pattern.

In [ ]:
lead_names = ['I','II','III','aVR','aVL','aVF','V1','V2','V3','V4','V5','V6']

normal_id  = df[df['label_merged']==0]['patient_id'].iloc[0]
brugada_id = df[df['label_merged']==1]['patient_id'].iloc[0]

def load_signal(pid):
    rec = wfdb.rdrecord(f'{DATA_DIR}/{str(pid)}/{str(pid)}')
    return rec.p_signal.T  # (12, 1200)

normal_sig  = load_signal(normal_id)
brugada_sig = load_signal(brugada_id)
t = np.arange(1200) / 100  # time in seconds

fig, axes = plt.subplots(12, 2, figsize=(16, 22))
for i, lead in enumerate(lead_names):
    axes[i,0].plot(t, normal_sig[i],  color='steelblue', lw=0.8)
    axes[i,1].plot(t, brugada_sig[i], color='crimson',   lw=0.8)
    axes[i,0].set_ylabel(lead, fontsize=8)
    for j in range(2):
        axes[i,j].tick_params(labelsize=7)
        axes[i,j].grid(alpha=0.3)

axes[0,0].set_title(f'Normal (ID: {normal_id})',  fontweight='bold', color='steelblue', fontsize=11)
axes[0,1].set_title(f'Brugada (ID: {brugada_id})', fontweight='bold', color='crimson',  fontsize=11)
axes[-1,0].set_xlabel('Time (s)')
axes[-1,1].set_xlabel('Time (s)')
plt.suptitle('12-Lead ECG: Normal vs. Brugada\n(Note ST-segment differences in V1–V3)',
             fontsize=12, fontweight='bold', y=1.005)
plt.tight_layout()
plt.savefig('../outputs/figures/eda_ecg_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Lead-Wise Amplitude Statistics by Class

Are there measurable amplitude differences between Normal and Brugada patients 
per lead? This motivates our decision to use all 12 leads rather than V1–V3 only.

In [ ]:
print('Computing per-lead statistics — takes ~2 minutes...')
stats = {'lead': [], 'class': [], 'std': []}

for _, row in df.iterrows():
    pid   = str(row['patient_id'])
    label = row['label_merged']
    try:
        sig = load_signal(pid)
        for i, lead in enumerate(lead_names):
            stats['lead'].append(lead)
            stats['class'].append('Brugada' if label else 'Normal')
            stats['std'].append(np.std(sig[i]))
    except:
        pass

stats_df = pd.DataFrame(stats)

plt.figure(figsize=(13, 5))
sns.boxplot(data=stats_df, x='lead', y='std', hue='class',
            palette={'Normal':'steelblue', 'Brugada':'crimson'})
plt.title('Signal Std Dev per Lead — Normal vs. Brugada', fontweight='bold', fontsize=12)
plt.xlabel('ECG Lead')
plt.ylabel('Std Dev (mV)')
plt.tight_layout()
plt.savefig('../outputs/figures/eda_lead_statistics.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA complete. Figures saved to outputs/figures/')